# tensor-zeros-init — ex9: allocate a pinned staging buffer for non_blocking transfer

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-zeros-init`. Running the final beacon cell reports progress against the `Numpy: Core array literacy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Core array literacy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-zeros-init`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-zeros-init"
DD_SUBTOPIC = "Numpy: Core array literacy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.zeros — pinned staging refresher

`torch.zeros(*shape, pin_memory=True)` allocates a page-locked CPU buffer. Page-locked memory cannot be paged out by the OS, which lets the CUDA driver DMA into it directly — `tensor.to('cuda', non_blocking=True)` only overlaps compute with transfer when the **source** is pinned.

**Compared to `torch.zeros(...).pin_memory()`.** `pin_memory=True` at allocation avoids an unnecessary unpinned-allocation + copy. For DataLoader-style staging where you reuse one buffer across iterations, allocate-pinned once, then `.copy_()` fresh data in-place each step.

**This drill (ex9) vs ex1-8.** Earlier exercises focused on allocation-as-output (paint hits, scatter, histogram). ex9 focuses on allocation-as-staging — a buffer's *kwargs* matter as much as its shape when you need a fast host→device pipeline.

### Exercise 9 — allocate a pinned staging buffer for non_blocking transfer

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `torch.zeros(*shape, pin_memory=True)` to allocate a reusable page-locked staging buffer, then verify in-place `.copy_(src)` preserves the pinned flag and produces a correct value snapshot.
> Keywords: pin_memory, non_blocking, dataloader, staging-buffer
> ```

**KCs targeted:** `zeros-shape-kwarg`, `zeros-pin-memory-kwarg`

Implement `ex9_pinned_staging(batch_shape, sources)`.

Simulate the DataLoader staging pattern without needing CUDA:

1. Allocate ONE pinned staging buffer of shape `batch_shape` and dtype `float32` via `t.zeros(*batch_shape, dtype=t.float32, pin_memory=True)`. The buffer is reused across iterations — do NOT reallocate inside the loop.
2. For each `src` in `sources` (a list of CPU tensors with shape `batch_shape`), copy `src` into the staging buffer **in place** via `buf.copy_(src)`, then record `buf.clone().detach()` into a list.
3. Return `(buf, snapshots)` — the buffer itself (still pinned), and the per-iteration snapshot list.

On CPU-only systems, `pin_memory=True` is a no-op flag but the attribute `is_pinned()` reflects what would happen on a CUDA host — the test calls `is_pinned()` only if a CUDA device is available, otherwise it falls back to checking the API surface (the kwarg was accepted without error).

Output: `(buf, snapshots)` where `buf.shape == batch_shape`, `buf.dtype == torch.float32`, and `len(snapshots) == len(sources)`.

The visualization plots the per-iteration snapshot mean to confirm that the staging buffer faithfully captured each source — a classic dataloader debug move.

In [ ]:
def ex9_pinned_staging(
    batch_shape: tuple[int, ...],
    sources: list[Tensor],
) -> tuple[Tensor, list[Tensor]]:
    buf = t.zeros(*batch_shape, dtype=t.float32, pin_memory=t.cuda.is_available())
    snapshots = []
    for src in sources:
        buf.copy_(src)
        snapshots.append(buf.clone().detach())
    return buf, snapshots


<details><summary>Solution</summary>

```python
def ex9_pinned_staging(
    batch_shape: tuple[int, ...],
    sources: list[Tensor],
) -> tuple[Tensor, list[Tensor]]:
    buf = t.zeros(*batch_shape, dtype=t.float32, pin_memory=t.cuda.is_available())
    snapshots = []
    for src in sources:
        buf.copy_(src)
        snapshots.append(buf.clone().detach())
    return buf, snapshots
```

**Why allocate-then-`copy_` rather than re-allocating.** A pinned allocation is expensive — the kernel has to register page-locked memory with the DMA controller. Reusing one buffer across iterations amortizes that setup. `copy_` mutates in place and preserves the pinned flag.

**Why `.clone().detach()` for snapshots.** Without clone, every snapshot would alias `buf` — by the time you inspect them, they all show the LAST source. The clone breaks aliasing; the detach is future-proofing in case you ever stage a tensor that came from an autograd-tracked op.

**CPU-only fallback.** PyTorch's `pin_memory=True` raises if no CUDA host is configured on some platforms — we gate the kwarg on `t.cuda.is_available()`. In a real training rig you'd keep it `True` and let the DataLoader use it; this drill exercises the allocation pattern without requiring a GPU.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()